# Ambiguous matches

Reads the results saved by `match_injections_with_hdrs.ipynb` for the settings in `TAG`
(`{CRITERION}_{ASSIGNMENT}`), so run that notebook first with the same settings.

1. **One injection, several posteriors**: the injection lies inside the credible region of more than one recovered source.
2. **One posterior, several injections**: more than one injection lies inside the credible region of a single recovered source.

In both plots the pair kept by the one-to-one assignment is highlighted. Only the six matching
parameters are shown (phase and polarization are not used for matching).

In [ ]:
# Where the data live (see copu_cat.config for the layout) and where plots are saved.
DATA_DIR = '../data'
PLOTS_DIR = '../plots'

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from copu_cat.corner_plots import corner_injection_vs_posteriors, corner_posterior_vs_injections, MATCH_PARAMS

TAG = '6d_max_snr'   # same CRITERION and ASSIGNMENT as in match_injections_with_hdrs.ipynb
match_table = pd.read_feather(f'{DATA_DIR}/matches_{TAG}.feather')
inj_summary = pd.read_feather(f'{DATA_DIR}/injection_summary_{TAG}.feather')
post_summary = pd.read_feather(f'{DATA_DIR}/posterior_summary_{TAG}.feather')
posterior_snr = post_summary.set_index('Name')['SNR'] if 'SNR' in post_summary else None
os.makedirs(f'{PLOTS_DIR}/ambiguous', exist_ok=True)

matched = match_table[match_table['matched']]
print('injections matched to >= 2 posteriors:', (matched.groupby('Candidate').size() >= 2).sum())
print('posteriors matched to >= 2 injections:', (matched.groupby('Name').size() >= 2).sum())

## 1. One injection matched to several posteriors

In [ ]:
multi_inj = inj_summary[inj_summary['n_posteriors_matched'] >= 2].sort_values('n_posteriors_matched', ascending=False)
print(multi_inj['n_posteriors_matched'].value_counts().sort_index().rename('number of injections'))
multi_inj[['Name', 'Frequency', 'SNR', 'n_posteriors_matched', 'assigned_posterior', 'status']].head(10)

In [ ]:
# a few examples: the most-matched injection, the loudest one, and a random one with 2 matches
rng = np.random.default_rng(1)
inj_examples = list(dict.fromkeys([
    multi_inj['Name'].iloc[0],
    multi_inj.sort_values('SNR', ascending=False)['Name'].iloc[0],
    rng.choice(multi_inj.loc[multi_inj['n_posteriors_matched'] == 2, 'Name'].to_numpy()),
]))
# or pick your own, e.g. inj_examples = ['14379849']
for name in inj_examples:
    fig = corner_injection_vs_posteriors(name, match_table, inj_summary, posterior_snr=posterior_snr, data_dir=DATA_DIR)
    fig.savefig(f'{PLOTS_DIR}/ambiguous/injection_{name}_{TAG}.pdf', bbox_inches='tight')
    plt.show()

## 2. One posterior matched to several injections
Yes, this can be probed directly: `n_injections_matched` in the match table counts, for each posterior,
how many injections lie inside its `THRESHOLD` credible region.

In [ ]:
multi_post = post_summary[post_summary['n_injections_matched'] >= 2].sort_values('n_injections_matched', ascending=False)
print(multi_post['n_injections_matched'].value_counts().sort_index().rename('number of posteriors'))
multi_post[['Name', 'Frequency', 'SNR', 'n_injections_matched', 'assigned_injection']].head(10)

In [ ]:
post_examples = list(dict.fromkeys([
    multi_post['Name'].iloc[0],
    rng.choice(multi_post.loc[multi_post['n_injections_matched'] == 2, 'Name'].to_numpy()),
]))
for name in post_examples:
    fig = corner_posterior_vs_injections(name, match_table, inj_summary, data_dir=DATA_DIR)
    fig.savefig(f'{PLOTS_DIR}/ambiguous/posterior_{name}_{TAG}.pdf', bbox_inches='tight')
    plt.show()